### Agent 的高级用法-错误处理机制

情况1: 设置为 True/False/固定字符串

举例1: handle_errors 设置为 True

In [24]:
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage, SystemMessage
import os
from langchain.agents import create_agent
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy, AutoStrategy
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langchain_tavily import TavilySearch
from dotenv import load_dotenv
from rich import print as rprint
from pydantic import BaseModel, Field
from typing import Union

load_dotenv(override=True)
ZHIPUAI_API_KEY = os.getenv("ZHIPUAI_API_KEY")
ZHIPUAI_API_BASE_URL = os.getenv("ZHIPUAI_API_BASE_URL")

# 1. 模型初始化
model = init_chat_model(
    model='glm-5.3',
    model_provider="openai",
    api_key=ZHIPUAI_API_KEY,
    base_url=ZHIPUAI_API_BASE_URL
)

In [25]:
class ContractInfo(BaseModel):
    """个人联系信息"""
    name: str = Field(description="姓名")
    email: str = Field(description="电子邮箱")

class EventDetails(BaseModel):
    """活动详情"""
    event_name: str = Field(description="活动名称")
    date: str = Field(description="活动日期")


agent = create_agent(
    model=model,
    response_format=ToolStrategy(
        Union[ContractInfo, EventDetails],
        tool_message_content="提取完成",
        handle_errors=True
        # handle_errors="请检查输入数据"
    )
)

response = agent.invoke({
    "messages": [{
        "role": "user",
        "content": f"请提取以下文本中内容: 姓名: 张三, 电子邮箱: zhang3@atguigu.com, 活动名称: 公司年会, 活动日期: 2026-07-15"
    }]
})

rprint(response)

{
    'messages': [
        HumanMessage(
            content='请提取以下文本中内容: 姓名: 张三, 电子邮箱: zhang3@atguigu.com, 活动名称: 公司年会, 活动日期: 
2026-07-15',
            additional_kwargs={},
            response_metadata={},
            id='63b7da42-1701-4a8c-a87f-208030ef2255'
        ),
        AIMessage(
            content='我将提取文本中的个人信息和活动详情:',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 169,
                    'prompt_tokens': 290,
                    'total_tokens': 459,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 111,
                        'rejected_prediction_tokens': None
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 256}
                },
                'model_provider': 'openai',
                'model_name': 'glm-5.3',
                'system_fingerprint': None,
                'id': '20261116144119a3dee8dc9ef64218',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a1e316-b702-7bf1-bfdf-5e87e3287ccf-0',
            tool_calls=[
                {
                    'name': 'ContractInfo',
                    'args': {'email': 'zhang3@atguigu.com', 'name': '张三'},
                    'id': 'call_-7079994033992364268',
                    'type': 'tool_call'
                },
                {
                    'name': 'EventDetails',
                    'args': {'date': '2026-07-15', 'event_name': '公司年会'},
                    'id': 'call_-7079994033992364267',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 290,
                'output_tokens': 169,
                'total_tokens': 459,
                'input_token_details': {'cache_read': 256},
                'output_token_details': {'reasoning': 111}
            }
        ),
        ToolMessage(
            content='Error: Model incorrectly returned multiple structured responses (ContractInfo, EventDetails) 
when only one is expected.\n Please fix your mistakes.',
            name='ContractInfo',
            id='c9c61e0a-97ff-415a-96e6-eee352da349e',
            tool_call_id='call_-7079994033992364268'
        ),
        ToolMessage(
            content='Error: Model incorrectly returned multiple structured responses (ContractInfo, EventDetails) 
when only one is expected.\n Please fix your mistakes.',
            name='EventDetails',
            id='79e7e154-cbea-414e-bb59-da2b47fe60bb',
            tool_call_id='call_-7079994033992364267'
        ),
        AIMessage(
            content='抱歉,看起来需要分别单独调用工具。让我先提取联系人信息:',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 87,
                    'prompt_tokens': 406,
                    'total_tokens': 493,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 46,
                        'rejected_prediction_tokens': None
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'glm-5.3',
                'system_fingerprint': None,
                'id': '20261116144123080948597bbc486b',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a1e316-c821-7cf3-a5ce-1d9571be2b84-0',
            tool_calls=[
                {
                    'name': 'Contra

举例2: handle_errors 设置为 False

In [26]:
class ContractInfo(BaseModel):
    """个人联系信息"""
    name: str = Field(description="姓名")
    email: str = Field(description="电子邮箱")

class EventDetails(BaseModel):
    """活动详情"""
    event_name: str = Field(description="活动名称")
    date: str = Field(description="活动日期")


agent = create_agent(
    model=model,
    response_format=ToolStrategy(
        Union[ContractInfo, EventDetails],
        tool_message_content="提取完成",
        # handle_errors=False
        handle_errors=True
    )
)

response = agent.invoke({
    "messages": [{
        "role": "user",
        "content": f"请提取以下文本中内容: 姓名: 张三, 电子邮箱: zhang3@atguigu.com, 活动名称: 公司年会, 活动日期: 2026-07-15"
    }]
})

rprint(response)

{
    'messages': [
        HumanMessage(
            content='请提取以下文本中内容: 姓名: 张三, 电子邮箱: zhang3@atguigu.com, 活动名称: 公司年会, 活动日期: 
2026-07-15',
            additional_kwargs={},
            response_metadata={},
            id='2834de8d-e497-421a-8858-d51630c1ef44'
        ),
        AIMessage(
            content='我将从文本中提取联系信息和活动详情：',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 159,
                    'prompt_tokens': 290,
                    'total_tokens': 449,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 100,
                        'rejected_prediction_tokens': None
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 256}
                },
                'model_provider': 'openai',
                'model_name': 'glm-5.3',
                'system_fingerprint': None,
                'id': '2026111614412890f7f8f91dbb44c1',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a1e316-dc13-71a0-8e9c-1cb3b1ea60dd-0',
            tool_calls=[
                {
                    'name': 'ContractInfo',
                    'args': {'email': 'zhang3@atguigu.com', 'name': '张三'},
                    'id': 'call_-7079998672557044417',
                    'type': 'tool_call'
                },
                {
                    'name': 'EventDetails',
                    'args': {'date': '2026-07-15', 'event_name': '公司年会'},
                    'id': 'call_-7079998672557044416',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 290,
                'output_tokens': 159,
                'total_tokens': 449,
                'input_token_details': {'cache_read': 256},
                'output_token_details': {'reasoning': 100}
            }
        ),
        ToolMessage(
            content='Error: Model incorrectly returned multiple structured responses (ContractInfo, EventDetails) 
when only one is expected.\n Please fix your mistakes.',
            name='ContractInfo',
            id='f3d92b3e-a634-4f74-baa0-c355db2b8552',
            tool_call_id='call_-7079998672557044417'
        ),
        ToolMessage(
            content='Error: Model incorrectly returned multiple structured responses (ContractInfo, EventDetails) 
when only one is expected.\n Please fix your mistakes.',
            name='EventDetails',
            id='87ee8c18-5fce-40cb-8209-9e3493034d80',
            tool_call_id='call_-7079998672557044416'
        ),
        AIMessage(
            content='系统提示一次只能提取一个结构化信息，让我分别进行提取。首先提取个人联系信息：',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 88,
                    'prompt_tokens': 407,
                    'total_tokens': 495,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 41,
                        'rejected_prediction_tokens': None
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 256}
                },
                'model_provider': 'openai',
                'model_name': 'glm-5.3',
                'system_fingerprint': None,
                'id': '202611161441321e503e16071748df',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a1e316-ec75-7772-95d8-3911712ca4e6-0',
            tool_calls=[
                {
                    'na

举例3: handle_errors 设置为固定的字符串

In [ ]:
class ContractInfo(BaseModel):
    """个人联系信息"""
    name: str = Field(description="姓名")
    email: str = Field(description="电子邮箱")

class EventDetails(BaseModel):
    """活动详情"""
    event_name: str = Field(description="活动名称")
    date: str = Field(description="活动日期")


agent = create_agent(
    model=model,
    response_format=ToolStrategy(
        Union[ContractInfo, EventDetails],
        tool_message_content="提取完成",
        handle_errors="请检查输入数据"
    )
)

response = agent.invoke({
    "messages": [{
        "role": "user",
        "content": f"请提取以下文本中内容: 姓名: 张三, 电子邮箱: zhang3@atguigu.com, 活动名称: 公司年会, 活动日期: 2026-07-15"
    }]
})

rprint(response)

### 情况2: 设置为指定异常类型

举例:

In [ ]:
from langchain.agents.structured_output import MultipleStructuredOutputsError, StructuredOutputValidationError

class ContractInfo(BaseModel):
    """个人联系信息"""
    name: str = Field(description="姓名")
    email: str = Field(description="电子邮箱")

class EventDetails(BaseModel):
    """活动详情"""
    event_name: str = Field(description="活动名称")
    date: str = Field(description="活动日期")


agent = create_agent(
    model=model,
    response_format=ToolStrategy(
        Union[ContractInfo, EventDetails],
        tool_message_content="提取完成",
        handle_errors=(MultipleStructuredOutputsError, StructuredOutputValidationError)
    )
)

response = agent.invoke({
    "messages": [{
        "role": "user",
        "content": f"请提取以下文本中内容: 姓名: 张三, 电子邮箱: zhang3@atguigu.com, 活动名称: 公司年会, 活动日期: 2026-07-15"
    }]
})

rprint(response)

### 情况3: 设置为自定义错误处理函数

In [ ]:
from langchain.agents.structured_output import MultipleStructuredOutputsError, StructuredOutputValidationError


# 自定义错误处理的函数
def custom_error_handler(error: Exception) -> str:
    """自定义错误处理"""
    error_str = str(error)
    print(f"捕获到的错误类型是: {type(error).__name__}")
    print(f"错误详情: {error_str}")

    if isinstance(error, MultipleStructuredOutputsError): 
        return "检查到多个响应, 请选择最相关的一个进行返回"
    elif isinstance(error, StructuredOutputValidationError): 
        return "数据格式有无, 请检查字段是否符合要求"
    else:
        return f"Error: {error_str}"

class ContractInfo(BaseModel):
    """个人联系信息"""
    name: str = Field(description="姓名")
    email: str = Field(description="电子邮箱")

class EventDetails(BaseModel):
    """活动详情"""
    event_name: str = Field(description="活动名称")
    date: str = Field(description="活动日期")


agent = create_agent(
    model=model,
    response_format=ToolStrategy(
        Union[ContractInfo, EventDetails],
        tool_message_content="提取完成",
        handle_errors=custom_error_handler
        # handle_errors="请检查输入数据"
    )
)

response = agent.invoke({
    "messages": [{
        "role": "user",
        "content": f"请提取以下文本中内容: 姓名: 张三, 电子邮箱: zhang3@atguigu.com, 活动名称: 公司年会, 活动日期: 2026-07-15"
    }]
})

rprint(response)